# scene-text-vision 런북 (Colab)

순서: 설치 → 런타임 재시작 → Drive 마운트 → 데이터 점검 → smoke → zero-shot 제출 → LoRA 학습 → 선택적 재추론/결합 → 최종 전체 학습.

`docs/DAY1_PLAN.md`와 같은 순서입니다. A100 80GB는 런타임 유형에서 GPU=A100, High-RAM 켜기. 산출물은 반드시 Drive(RUNS)에 저장합니다.

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchao
!pip install -q -U git+https://github.com/huggingface/transformers
!pip install -q -U "peft>=0.20.0" "accelerate>=1.8.0" "bitsandbytes>=0.46.1" "pandas>=2.2" "scikit-learn>=1.5" pillow tqdm safetensors
print('설치 완료 -> 런타임 > 세션 다시 시작 후 다음 셀부터 실행')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# 데이터와 산출물은 Drive에, 코드는 레포에서 가져온다.
ROOT  = '/content/drive/MyDrive/scene-text-vision-data'   # train.csv, test.csv, sample_submission.csv, 이미지 폴더
RUNS  = '/content/drive/MyDrive/scene-text-vision-runs'   # adapter, 확률 npz, 제출 csv (세션이 끊겨도 남는다)
MODEL = 'Qwen/Qwen3.5-27B'                                # GPU가 작으면 Qwen/Qwen3.5-9B, Qwen/Qwen3.5-4B
os.makedirs(RUNS, exist_ok=True)

# 비공개 레포라 토큰이 필요하다: https://<GITHUB_TOKEN>@github.com/... 형태로 바꾸거나,
# scripts/ 폴더를 Drive에 복사한 뒤 CODE를 그 경로로 바꾼다.
!git clone https://github.com/min9-less-min9-team/scene-text-vision-repo.git /content/scene-text-vision
CODE = '/content/scene-text-vision/scripts'
os.chdir(CODE)
!ls -la $CODE $ROOT | head -30

## 1. 데이터 점검 (GPU 불필요)

In [ ]:
!python inspect_data.py --root $ROOT
!python vqa_textmc.py --mode dryrun --root $ROOT --out-dir $RUNS/dry

## 2. Smoke test (16개 학습 + 8개 추론, 버전/OOM 확인)

In [ ]:
!python vqa_textmc.py --mode smoke --root $ROOT --model-id $MODEL --copy-local /content/data --out-dir $RUNS/smoke

## 3. Zero-shot: 검증 400개 정확도 + 첫 제출 파일

해상도 비교가 필요하면 `--max-visual-tokens`를 1024/1536으로 바꿔 검증 정확도만 비교(`--max-test-samples 400`).

In [ ]:
!python vqa_textmc.py --mode zeroshot --root $ROOT --model-id $MODEL --copy-local /content/data \n    --tta-perms 2 --max-visual-tokens 1280 --out-dir $RUNS/zs
# 제출: $RUNS/zs/submission_zeroshot.csv

## 4. LoRA 학습 (검증 분할 10%) → 검증 리포트 → test TTA4

In [ ]:
!python vqa_textmc.py --mode all --root $ROOT --model-id $MODEL --copy-local /content/data \n    --epochs 1 --lr 1e-4 --lora-r 16 --lora-alpha 32 --lora-dropout 0.05 \n    --batch-size 1 --grad-accum 8 --max-visual-tokens 1280 --valid-size 0.1 \n    --tta-perms 4 --log-every 50 --out-dir $RUNS/ft_v1
# 제출: $RUNS/ft_v1/submission_test.csv | 오답 분석: $RUNS/ft_v1/valid_predictions.csv

## 5. 오답 분석 → 불확실 문항 선택

In [ ]:
import pandas as pd, numpy as np
v = pd.read_csv(f'{RUNS}/ft_v1/valid_predictions.csv')
v['ok'] = v['answer'] == v['gold']
print('valid acc', v['ok'].mean())
print(v.groupby('qtype')['ok'].agg(['mean','count']))
print(v[~v['ok']][['question','answer','gold','margin','qtype']].head(30).to_string())

t = pd.read_csv(f'{RUNS}/ft_v1/test_predictions_detailed.csv')
unc = t[(t['margin'] < 0.15) | (t['unc_score'] >= t['unc_score'].quantile(0.8))]
print('uncertain test items:', len(unc))
open(f'{RUNS}/ft_v1/uncertain_ids.txt','w').write('
'.join(unc['id'].astype(str)))

## 6. 불확실 문항만 재추론 (고해상도 / 멀티크롭 / LoRA OFF) → gated 결합

검증셋에서 같은 절차로 효과를 먼저 확인할 것(`--mode infer`는 test 대상이므로 검증은 4절 실행에서 `--max-visual-tokens`만 바꿔 비교).

In [ ]:
!python vqa_textmc.py --mode infer --root $ROOT --model-id $MODEL --copy-local /content/data \n    --adapter $RUNS/ft_v1/adapter --subset-ids $RUNS/ft_v1/uncertain_ids.txt \n    --max-visual-tokens 1536 --tta-perms 4 --out-dir $RUNS/ft_v1_hires

!python vqa_textmc.py --mode infer --root $ROOT --model-id $MODEL --copy-local /content/data \n    --adapter $RUNS/ft_v1/adapter --subset-ids $RUNS/ft_v1/uncertain_ids.txt \n    --crop-mode five --tta-perms 4 --out-dir $RUNS/ft_v1_crop

!python vqa_textmc.py --mode infer --root $ROOT --model-id $MODEL --copy-local /content/data \n    --adapter $RUNS/ft_v1/adapter --subset-ids $RUNS/ft_v1/uncertain_ids.txt \n    --lora-off --tta-perms 4 --out-dir $RUNS/ft_v1_loraoff

In [ ]:
!python blend_probs.py --primary $RUNS/ft_v1/test_probs.npz --secondary $RUNS/ft_v1_hires/test_subset_probs.npz \n    --strategy gated --gate-margin 0.15 --w 0.35 --sample-csv $ROOT/sample_submission.csv --out $RUNS/sub_hires_gated.csv

!python blend_probs.py --primary $RUNS/sub_hires_gated.npz --secondary $RUNS/ft_v1_loraoff/test_subset_loraoff_probs.npz \n    --strategy gated --gate-margin 0.15 --w 0.5 --min-secondary-conf 0.60 --min-secondary-agree 0.75 --max-support-gap 0.12 \n    --sample-csv $ROOT/sample_submission.csv --out $RUNS/sub_hires_baseoff.csv

## 7. (선택) OCR 힌트 브랜치

In [ ]:
!pip install -q easyocr
!python ocr_extract.py --root $ROOT --split test --out $RUNS/ocr_test.csv
!python vqa_textmc.py --mode infer --root $ROOT --model-id $MODEL --copy-local /content/data \n    --adapter $RUNS/ft_v1/adapter --ocr-csv $RUNS/ocr_test.csv --tta-perms 4 --out-dir $RUNS/ft_v1_ocr

## 8. 최종: 전체 train으로 재학습 (검증 분할 없음) → 검증된 결합 규칙 그대로 적용

In [ ]:
!python vqa_textmc.py --mode all --root $ROOT --model-id $MODEL --copy-local /content/data \n    --epochs 1 --lr 1e-4 --max-visual-tokens 1280 --valid-size 0 --tta-perms 4 --out-dir $RUNS/ft_full